# SFT->DPO ChartQA holdout evalGreedy generation on the 30-question ChartQA holdout using the SFT->DPO adapter (method 7/7).

In [ ]:
import subprocess, sys

gpu_names = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True, text=True
).stdout.strip().splitlines()
print('Detected GPUs (nvidia-smi):', gpu_names)
is_p100 = any('P100' in n for n in gpu_names)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)

if is_p100:
    print('*** Tesla P100 detected. Installing the validated older stack...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow'], check=True)
else:
    print('Non-P100 GPU: upgrading transformers to a current release, leaving torch/peft/accelerate at image defaults.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.49.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'peft==0.14.0', 'qwen-vl-utils==0.0.14'], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, transformers: {transformers.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')


In [ ]:
import os
from pathlib import Path
import subprocess

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

import glob

print('Contents of /kaggle/input:', os.listdir('/kaggle/input'))
matches = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
print('Found adapter_config.json at:', matches)
assert len(matches) == 1, f'Expected exactly 1 adapter under /kaggle/input, found {len(matches)}: {matches}'
ADAPTER_PATH = os.path.dirname(matches[0])
print('Using adapter:', ADAPTER_PATH)
print('Adapter files:', sorted(os.listdir(ADAPTER_PATH)))


In [ ]:
import json
import re
from pathlib import Path

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

import sys
sys.path.insert(0, str(Path('/tmp/chart-prm/src')))
from chart_prm.generator import build_generation_prompt

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SPLIT_PATH = Path('data/splits/chartqa_eval_ids.json')
QUESTIONS_PATH = Path('data/ChartQA/data/chartqa_reasoning.json')
IMAGES_DIR = Path('data/ChartQA/images')
SYSTEM = 'sft_dpo'
OUTPUT_PATH = Path(f'/kaggle/working/{SYSTEM}_chartqa_holdout_generations.jsonl')
SUMMARY_PATH = Path(f'/kaggle/working/{SYSTEM}_chartqa_holdout_accuracy.json')

with SPLIT_PATH.open(encoding='utf-8') as f:
    eval_ids = [str(x) for x in json.load(f)]
with QUESTIONS_PATH.open(encoding='utf-8') as f:
    question_data = json.load(f)
print(f'Holdout size: {len(eval_ids)}')

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, attn_implementation='sdpa',
    device_map={'': 0}, low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, adapter_name='sft_dpo')
model.set_adapter('sft_dpo')
model.eval()
print('Model ready: sft_dpo')

FINAL_ANSWER_RES = [
    re.compile(r'Final Answer:\s*(.+)', re.IGNORECASE | re.DOTALL),
    re.compile(r'\*\*Final Answer\*\*:\s*(.+)', re.IGNORECASE | re.DOTALL),
    re.compile(r'Therefore,? the answer is:?\s*(.+)', re.IGNORECASE | re.DOTALL),
]

def extract_final_answer(text):
    for pattern in FINAL_ANSWER_RES:
        matches = pattern.findall(text or '')
        if matches:
            return matches[-1].strip().splitlines()[0].strip().strip('\"\'`')
    return ''

def normalize_answer(text):
    return re.sub(r'\s+', ' ', (text or '').strip().lower())

def generate_response(image_path, question):
    prompt = build_generation_prompt(question)
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': str(image_path)},
        {'type': 'text', 'text': prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to('cuda')
    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False, use_cache=True)
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

done_ids = set()
if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                done_ids.add(json.loads(line)['question_id'])
print(f'Resuming with {len(done_ids)} completed questions.')

with OUTPUT_PATH.open('a', encoding='utf-8') as handle:
    for idx, qid in enumerate(eval_ids):
        if qid in done_ids:
            continue
        record = question_data[qid]
        image_path = IMAGES_DIR / f'{qid}.jpg'
        assert image_path.exists(), f'Missing image for {qid}'

        response = generate_response(image_path, record['query'])

        row = {
            'question_id': qid,
            'question': record['query'],
            'ground_truth': record['answer'],
            'responses': {SYSTEM: response},
            'predicted_answers': {SYSTEM: extract_final_answer(response)},
        }
        if idx == 0:
            if not response.strip():
                raise RuntimeError(f'{SYSTEM} produced an empty generation on id={qid}. Aborting: likely collapsed LoRA or load failure.')
            print(f'Smoke OK for id={qid}; sample response: {response[:200]!r}')
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')
        handle.flush()
        done_ids.add(qid)
        if (idx + 1) % 5 == 0 or idx == 0:
            print(f'[{idx + 1}/{len(eval_ids)}] completed question_id={qid}')

print(f'Saved generations to {OUTPUT_PATH}')


In [ ]:
rows = []
with OUTPUT_PATH.open(encoding='utf-8') as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
assert len(rows) == len(eval_ids), f'Expected {len(eval_ids)} rows, found {len(rows)}'

correct = 0
extracted = 0
for row in rows:
    pred = row['predicted_answers'].get(SYSTEM, '')
    if pred:
        extracted += 1
    if normalize_answer(pred) == normalize_answer(row['ground_truth']):
        correct += 1

summary = {
    'n': len(rows),
    'exact_match': {SYSTEM: {'correct': correct, 'accuracy': correct / len(rows)}},
    'extracted_answer_rate': {SYSTEM: extracted / len(rows)},
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
